# Feature Cache Notebook

Build 91-day window features once, save them to disk, and reuse the cached matrices for training experiments. The default config builds a weather-only cache with `score_*` features disabled.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from make_features import (
    HORIZONS,
    PRED_COLS,
    align_feature_columns,
    build_test_features,
    build_training_set_for_horizon,
    read_test_csv,
    read_train_csv,
)
from train import make_model

PROJECT_ROOT

## Config

Use `INCLUDE_SCORE_FEATURES = False` for the clean weather-only experiment. Set it to `True` to use the corrected score-history features, which only look before the 91-day weather window starts.

In [ ]:
TRAIN_CSV = PROJECT_ROOT / "data" / "train.csv"
TEST_CSV = PROJECT_ROOT / "data" / "test.csv"

INCLUDE_SCORE_FEATURES = False
STRIDE = 8
MAX_REGIONS = None
MAX_TRAIN_EXAMPLES_PER_HORIZON = 100_000

CACHE_NAME = "features_score" if INCLUDE_SCORE_FEATURES else "features_no_score"
FEATURE_DIR = PROJECT_ROOT / CACHE_NAME
MODEL_DIR = PROJECT_ROOT / ("models_score_cached" if INCLUDE_SCORE_FEATURES else "models_no_score_cached")

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

{
    "feature_dir": str(FEATURE_DIR),
    "model_dir": str(MODEL_DIR),
    "include_score_features": INCLUDE_SCORE_FEATURES,
    "stride": STRIDE,
    "max_regions": MAX_REGIONS,
    "max_train_examples_per_horizon": MAX_TRAIN_EXAMPLES_PER_HORIZON,
}

## Read Data

In [ ]:
train_df = read_train_csv(TRAIN_CSV, max_regions=MAX_REGIONS)
test_df = read_test_csv(TEST_CSV, max_regions=MAX_REGIONS)

print("train", train_df.shape, "regions", train_df["region_id"].nunique())
print("test ", test_df.shape, "regions", test_df["region_id"].nunique())
print("labels", int(train_df["score"].notna().sum()))

## Build And Cache Training Features

This writes one file per horizon. Each file includes metadata columns, target `y`, and feature columns.

In [ ]:
feature_columns_by_horizon = {}
cache_summary = []

for h in HORIZONS:
    print(f"building horizon {h}")
    X, y, meta = build_training_set_for_horizon(
        train_df,
        horizon=h,
        stride=STRIDE,
        max_train_examples=MAX_TRAIN_EXAMPLES_PER_HORIZON,
        include_score_features=INCLUDE_SCORE_FEATURES,
    )
    feature_columns = list(X.columns)
    feature_columns_by_horizon[str(h)] = feature_columns

    cached = pd.concat([meta.reset_index(drop=True), pd.Series(y, name="y"), X.reset_index(drop=True)], axis=1)
    out_path = FEATURE_DIR / f"train_h{h}.pkl"
    cached.to_pickle(out_path)

    cache_summary.append({"horizon": h, "rows": len(cached), "features": len(feature_columns), "path": str(out_path)})
    print(cache_summary[-1])

pd.DataFrame(cache_summary)

## Build And Cache Test Features

In [ ]:
history_df = train_df if INCLUDE_SCORE_FEATURES else None
X_test, test_meta = build_test_features(test_df, history_df, include_score_features=INCLUDE_SCORE_FEATURES)
test_cached = pd.concat([test_meta.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
test_path = FEATURE_DIR / "test.pkl"
test_cached.to_pickle(test_path)

cache_metadata = {
    "include_score_features": INCLUDE_SCORE_FEATURES,
    "score_history_timing": "window_start" if INCLUDE_SCORE_FEATURES else "disabled",
    "stride": STRIDE,
    "max_regions": MAX_REGIONS,
    "max_train_examples_per_horizon": MAX_TRAIN_EXAMPLES_PER_HORIZON,
    "feature_columns_by_horizon": feature_columns_by_horizon,
    "train_files": {str(row["horizon"]): row["path"] for row in cache_summary},
    "test_file": str(test_path),
}
(FEATURE_DIR / "metadata.json").write_text(json.dumps(cache_metadata, indent=2), encoding="utf-8")

print("test", test_cached.shape, test_path)
print("metadata", FEATURE_DIR / "metadata.json")

## Inspect Cache

In [ ]:
h = 1
cached = pd.read_pickle(FEATURE_DIR / f"train_h{h}.pkl")
score_cols = [c for c in cached.columns if c.startswith("score_")]

print(cached.shape)
print("score feature columns:", score_cols)
cached.head()

## Train From Cache

This cell trains one LightGBM model per horizon from cached features. Re-run this cell freely while tuning model parameters; feature generation will not run again.

In [ ]:
model_metadata = {
    "model_kind_by_horizon": {},
    "feature_columns_by_horizon": feature_columns_by_horizon,
    "stride": STRIDE,
    "max_regions": MAX_REGIONS,
    "max_train_examples_per_horizon": MAX_TRAIN_EXAMPLES_PER_HORIZON,
    "include_score_features": INCLUDE_SCORE_FEATURES,
    "score_history_timing": "window_start" if INCLUDE_SCORE_FEATURES else "disabled",
    "target_mean": float(np.nanmean(train_df["score"].to_numpy(dtype=np.float32))),
}

for h in HORIZONS:
    cached = pd.read_pickle(FEATURE_DIR / f"train_h{h}.pkl")
    feature_columns = feature_columns_by_horizon[str(h)]
    X = align_feature_columns(cached[feature_columns], feature_columns)
    y = cached["y"].to_numpy(dtype=np.float32)

    model, model_kind = make_model(random_state=42 + h)
    print(f"training horizon {h}: {len(X)} rows x {X.shape[1]} features")
    model.fit(X, y)
    joblib.dump(model, MODEL_DIR / f"horizon_{h}.joblib")
    model_metadata["model_kind_by_horizon"][str(h)] = model_kind

(MODEL_DIR / "metadata.json").write_text(json.dumps(model_metadata, indent=2), encoding="utf-8")
print("saved", MODEL_DIR)

## Predict From Cached Test Features

In [ ]:
sample = pd.read_csv(PROJECT_ROOT / "sample_submission.csv")
test_cached = pd.read_pickle(FEATURE_DIR / "test.pkl")
pred_df = pd.DataFrame({"region_id": test_cached["region_id"].astype(str)})

for h, col in zip(HORIZONS, PRED_COLS):
    feature_columns = feature_columns_by_horizon[str(h)]
    Xh = align_feature_columns(test_cached[feature_columns], feature_columns)
    model = joblib.load(MODEL_DIR / f"horizon_{h}.joblib")
    pred_df[col] = np.clip(model.predict(Xh), 0.0, 5.0)

submission = sample[["region_id"]].merge(pred_df, on="region_id", how="left")
missing = submission[PRED_COLS].isna().any(axis=1)
if missing.any():
    fallback = float(model_metadata["target_mean"])
    submission.loc[missing, PRED_COLS] = fallback
submission = submission[sample.columns]

out_path = PROJECT_ROOT / "submissions" / ("submission_score_cached.csv" if INCLUDE_SCORE_FEATURES else "submission_no_score_cached.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)
print(out_path, submission.shape)
submission.head()